# 01 – Data understanding & cleaning
Look at the raw (messy) data, find the problems, then clean it with `clean_data()` from `src/preprocessing.py` (the same function `train.py` and `predict.py` use).

In [44]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

from preprocessing import clean_data

## 1. Load the raw data

In [45]:
df = pd.read_csv('../Data/raw/credit_card_fraud_messy.csv')

In [46]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   transaction_id            15000 non-null  str    
 1   customer_id               15000 non-null  str    
 2   transaction_datetime      15000 non-null  str    
 3   amount                    15000 non-null  float64
 4   merchant_category         15000 non-null  str    
 5   payment_channel           15000 non-null  str    
 6   card_type                 15000 non-null  str    
 7   card_present              15000 non-null  int64  
 8   device_type               15000 non-null  str    
 9   customer_age              15000 non-null  float64
 10  account_age_days          15000 non-null  float64
 11  avg_txn_amount_30d        15000 non-null  float64
 12  home_country              15000 non-null  str    
 13  transaction_country       15000 non-null  str    
 14  distance_from_hom

In [47]:
df.head()

,transaction_id,customer_id,transaction_datetime,amount,merchant_category,payment_channel,card_type,card_present,device_type,customer_age,...,transaction_country,distance_from_home_km,num_txn_last_24h,time_since_last_txn_min,failed_attempts_last_24h,is_fraud,hour,day,month,day_of_week
0,t114728,c11768,2026-01-11 13:38:04,210.32,fuel,in-store,prepaid,1,mobile,42.0,...,egypt,23.5,2,643.3,0,0,13,11,1,6
1,t105521,c12200,2026-03-30 01:04:15,174.08,entertainment,atm,debit,1,pos terminal,44.0,...,canada,60.2,1,794.3,0,0,1,30,3,0
2,t102368,c13304,2026-02-09 21:32:57,54.18,health,online,prepaid,0,unknown,45.0,...,usa,5.5,4,388.3,0,0,21,9,2,0
3,t109758,c11900,2026-03-05 04:17:48,32.19,gift cards,atm,credit,1,pos terminal,48.0,...,canada,11.0,7,255.9,1,1,4,5,3,3
4,t101925,c12179,2026-05-12 05:10:12,44.94,online services,online,debit,0,mobile,46.0,...,egypt,22.3,6,27.7,3,1,5,12,5,1


## 2. Find the problems

In [48]:
df.isnull().sum()

transaction_id              0
customer_id                 0
transaction_datetime        0
amount                      0
merchant_category           0
payment_channel             0
card_type                   0
card_present                0
device_type                 0
customer_age                0
account_age_days            0
avg_txn_amount_30d          0
home_country                0
transaction_country         0
distance_from_home_km       0
num_txn_last_24h            0
time_since_last_txn_min     0
failed_attempts_last_24h    0
is_fraud                    0
hour                        0
day                         0
month                       0
day_of_week                 0
dtype: int64

In [49]:
df.duplicated().sum()

np.int64(0)

In [50]:
# messy text values (casing, spaces, different spellings of the same thing)
for col in ['merchant_category', 'payment_channel', 'home_country', 'transaction_country']:
    print(col, sorted(df[col].dropna().unique()), '\n')

merchant_category ['clothing', 'electronics', 'entertainment', 'fuel', 'gift cards', 'grocery', 'health', 'jewelry', 'online services', 'restaurant', 'travel', 'unknown'] 

payment_channel ['atm', 'contactless', 'in store', 'in-store', 'online'] 

home_country ['canada', 'egypt', 'germany', 'uae', 'united kingdom', 'usa'] 

transaction_country ['brazil', 'canada', 'china', 'egypt', 'france', 'germany', 'india', 'nigeria', 'russia', 'turkey', 'uae', 'united kingdom', 'unknown', 'usa', 'vietnam'] 



In [51]:
# amount is stored as text because some values look like "$1,234.50"
print('amounts with a $ sign:', df['amount'].astype(str).str.contains('$', regex=False).sum())
df.loc[df['amount'].astype(str).str.contains('$', regex=False), 'amount'].head()

amounts with a $ sign: 0


Series([], Name: amount, dtype: float64)

In [52]:
# dates come in several formats
print(df['transaction_datetime'].str.len().value_counts())
df['transaction_datetime'].sample(8, random_state=1)

transaction_datetime
19    15000
Name: count, dtype: int64


7576     2026-05-10 01:19:56
10509    2026-04-12 20:41:21
4253     2026-02-24 12:48:31
5150     2026-03-22 01:21:19
506      2026-05-08 18:52:43
1622     2026-01-27 20:43:00
13684    2026-01-25 07:35:38
2145     2026-02-14 03:36:04
Name: transaction_datetime, dtype: str

In [53]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
amount,15000.0,556.242440,20438.657529,-50.0,25.7775,46.79,84.1800,1000000.00
card_present,15000.0,0.664333,0.472239,0.0,0.0000,1.00,1.0000,1.00
customer_age,15000.0,40.144133,12.255798,18.0,31.0000,40.00,48.0000,85.00
account_age_days,15000.0,1307.111333,1083.366311,3.0,650.0000,1108.00,1742.0000,30000.00
avg_txn_amount_30d,15000.0,58.746115,37.779930,8.0,33.9175,49.69,72.6825,574.65
distance_from_home_km,15000.0,465.100253,2612.012944,0.0,6.4000,13.30,25.3250,99999.00
num_txn_last_24h,15000.0,1.997000,4.787050,0.0,1.0000,2.00,3.0000,194.00
time_since_last_txn_min,15000.0,380.831393,402.430294,0.0,98.5000,255.90,521.4500,3972.90
failed_attempts_last_24h,15000.0,0.251733,0.597875,0.0,0.0000,0.00,0.0000,7.00
is_fraud,15000.0,0.089467,0.285426,0.0,0.0000,0.00,0.0000,1.00


In [54]:
# IQR check: only to LOOK at outliers, not to delete them.
# Many fraud rows are "outliers" (e.g. failed_attempts_last_24h), so deleting them would delete the fraud.
num_columns = df.select_dtypes(include=np.number).columns.drop('is_fraud')

for col in num_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"\n{col}")
    print("Outliers:", len(outliers))
    print("Lower:", lower)
    print("Upper:", upper)


amount
Outliers: 1250
Lower: -61.82625
Upper: 171.78375

card_present
Outliers: 0
Lower: -1.5
Upper: 2.5

customer_age
Outliers: 89
Lower: 5.5
Upper: 73.5

account_age_days
Outliers: 464
Lower: -988.0
Upper: 3380.0

avg_txn_amount_30d
Outliers: 759
Lower: -24.229999999999997
Upper: 130.83

distance_from_home_km
Outliers: 1551
Lower: -21.987499999999997
Upper: 53.71249999999999

num_txn_last_24h
Outliers: 279
Lower: -2.0
Upper: 6.0

time_since_last_txn_min
Outliers: 818
Lower: -535.9250000000001
Upper: 1155.875

failed_attempts_last_24h
Outliers: 2871
Lower: 0.0
Upper: 0.0

hour
Outliers: 0
Lower: -13.0
Upper: 35.0

day
Outliers: 0
Lower: -14.5
Upper: 45.5

month
Outliers: 107
Lower: -2.5
Upper: 9.5

day_of_week
Outliers: 0
Lower: -5.0
Upper: 11.0


In [55]:
# impossible values (not just unusual ones) -> these are the real errors
print('age <18 or >100       :', ((df.customer_age < 18) | (df.customer_age > 100)).sum())
print('account_age_days <0   :', (df.account_age_days < 0).sum())
print('account_age_days >10000:', (df.account_age_days > 10000).sum())
print('distance <0           :', (df.distance_from_home_km < 0).sum())
print('distance >20000       :', (df.distance_from_home_km > 20000).sum())
print('num_txn_last_24h >50  :', (df.num_txn_last_24h > 50).sum())

age <18 or >100       : 0
account_age_days <0   : 0
account_age_days >10000: 7
distance <0           : 0
distance >20000       : 15
num_txn_last_24h >50  : 15


In [56]:
print(df['is_fraud'].value_counts())
print(df['is_fraud'].value_counts(normalize=True).round(3))

is_fraud
0    13658
1     1342
Name: count, dtype: int64
is_fraud
0    0.911
1    0.089
Name: proportion, dtype: float64


## 3. Cleaning decisions (all implemented in `clean_data()`)
- drop exact duplicate rows (training only)
- strip + lowercase text, unify country names (`us`, `u.s.a`, `united states` → `usa`, ...), `in store` → `in-store`
- `amount`: remove `$` and `,` then convert to number
- impossible values (age outside 18-100, negative or huge account age / distance / amount, > 50 transactions in 24h) → NaN
- parse the mixed date formats → `hour`, `day_of_week` (+ `day`, `month`)
- extra features: `is_night`, `amount_ratio`, `is_foreign`
- missing values are **not** filled here: the pipeline in `train.py` fills them with the train-set medians / `unknown`

## 4. Clean

In [57]:
df_clean = clean_data(df, drop_duplicates=True)

## 5. Check the result

In [58]:
print("Dataset Shape:", df_clean.shape)
print("\nMissing Values:")
print(df_clean.isnull().sum())
print("\nDuplicates:")
print(df_clean.duplicated().sum())
print("\nData Types:")
print(df_clean.dtypes)

Dataset Shape: (15000, 26)

Missing Values:
transaction_id               0
customer_id                  0
transaction_datetime         0
amount                      20
merchant_category            0
payment_channel              0
card_type                    0
card_present                 0
device_type                  0
customer_age                 0
account_age_days             7
avg_txn_amount_30d           0
home_country                 0
transaction_country          0
distance_from_home_km       15
num_txn_last_24h            15
time_since_last_txn_min      0
failed_attempts_last_24h     0
is_fraud                     0
hour                         0
day                          0
month                        0
day_of_week                  0
is_night                     0
amount_ratio                20
is_foreign                   0
dtype: int64

Duplicates:
0

Data Types:
transaction_id                  str
customer_id                     str
transaction_datetime            str
a

In [59]:
for col in ['merchant_category', 'payment_channel', 'home_country', 'transaction_country']:
    print(col, sorted(df_clean[col].dropna().unique()), '\n')

merchant_category ['clothing', 'electronics', 'entertainment', 'fuel', 'gift cards', 'grocery', 'health', 'jewelry', 'online services', 'restaurant', 'travel', 'unknown'] 

payment_channel ['atm', 'contactless', 'in-store', 'online'] 

home_country ['canada', 'egypt', 'germany', 'uae', 'united kingdom', 'usa'] 

transaction_country ['brazil', 'canada', 'china', 'egypt', 'france', 'germany', 'india', 'nigeria', 'russia', 'turkey', 'uae', 'united kingdom', 'unknown', 'usa', 'vietnam'] 



In [60]:
df_clean[['amount','customer_age','account_age_days','distance_from_home_km','num_txn_last_24h']].describe().T

,count,mean,std,min,25%,50%,75%,max
amount,14980.0,76.361056,120.309859,1.02,25.7975,46.79,84.0525,5173.77
customer_age,15000.0,40.144133,12.255798,18.0,31.0,40.0,48.0,85.0
account_age_days,14993.0,1293.715067,888.62134,3.0,650.0,1108.0,1741.0,6000.0
distance_from_home_km,14985.0,402.169423,1481.881313,0.0,6.4,13.3,25.3,18409.9
num_txn_last_24h,14985.0,1.862262,1.622752,0.0,1.0,2.0,3.0,13.0


## 6. Save

In [61]:
df_clean.to_csv('../Data/processed/cleaned.csv', index=False)